# Data Understanding

In [ ]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader

# Karena file notebook ini ada di dalam folder 'notebooks/', 
# kita mundur satu direktori (../) untuk masuk ke folder 'data/'
pdf_folder_path = "../data/raw_pdfs/"

print(f"Mencari file PDF di direktori: {pdf_folder_path}...")

# Inisialisasi loader
loader = PyPDFDirectoryLoader(pdf_folder_path)

# Memuat dan mengekstrak teks dari semua PDF
pdf_docs = loader.load()

print(f"Selesai! Berhasil mengekstrak total {len(pdf_docs)} halaman dari seluruh PDF.")

# Mari kita intip sedikit hasil ekstraksinya
if len(pdf_docs) > 0:
    print("\n=== Cuplikan Teks Halaman Pertama ===")
    # Menampilkan 500 karakter pertama agar output tidak terlalu panjang
    print(pdf_docs[0].page_content[:500] + "...\n")
    print("=====================================")
    print(f"Sumber Metadata: {pdf_docs[0].metadata}")
else:
    print("Peringatan: Tidak ada teks yang berhasil diekstrak. Cek kembali folder raw_pdfs/.")

C:\Users\HP\AppData\Local\Temp\ipykernel_21356\627999312.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


Mencari file PDF di direktori: ../data/raw_pdfs/...
Selesai! Berhasil mengekstrak total 408 halaman dari seluruh PDF.

=== Cuplikan Teks Halaman Pertama ===
KEMENTERIAN PENDIDIKAN TINGGI, 
SAINS DAN TEKNOLOGI 
UNIVERSITAS HALU OLEO 
Kampus Hijau Bumi Tridharma, Anduonuhu, Jalan H.E.A. Mokodompit 
Telepon/Fax (0401) 3190006, Kendari 93232 
Laman : www.uho.ac.id 
SURAT EDARAN 
NOMOR 03/UN29/2026 
TENTANG 
PENYESUAIAN SISTEM KERJA KEDINASAN 
PADA BULAN SUCI RAMADHAN 1447 HIJRIAH 
SERTA DALAM RANGKA LIBUR NASIONAL DAN CUTI BERSAMA 
HARI RAYA NYEPI (TAHUN BARU SAKA 1948) 
DAN HARI RAYA IDUL FITRI 1447 HIJRIAH 
DI LINGKUNGAN KEMENTERIAN PENDIDIKAN TINGGI,...

Sumber Metadata: {'producer': 'Scanner System Image Conversion', 'creator': 'Scanner System', 'creationdate': '2026-02-19T08:47:38+08:00', 'moddate': '2026-02-19T08:47:38+08:00', 'source': '..\\data\\raw_pdfs\\03-SURAT-EDARAN-TENTANG-PENYESUAIAN-SISTEM-KERJA-KEDINASAN-BULAN-SUCI-RAMADHAN-LINGKUP-UHO.pdf', 'total_pages': 3, 'page': 0, 'p

# Load url path dan faq

In [2]:
import re
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import PyPDFLoader
# PERBAIKAN 1: Import Document dipindah ke langchain_core
from langchain_core.documents import Document

# =======================================================
# BAGIAN 1: BACA ISI WEBSITE DARI URL 
# =======================================================
url_file_path = "../data/custom_data/urls.txt"

try:
    with open(url_file_path, "r", encoding="utf-8") as f:
        urls = [line.strip() for line in f.readlines() if line.strip().startswith("http")]

    print(f"Ditemukan {len(urls)} URL. Mulai scraping isi website...")
    web_loader = WebBaseLoader(web_paths=urls)
    web_docs = web_loader.load()
    print("Selesai scraping website!")
except FileNotFoundError:
    print(f"Error: File '{url_file_path}' tidak ditemukan. Pastikan sudah membuat file urls.txt.")
    web_docs = []


# =======================================================
# BAGIAN 2: LOAD FAQ DARI PDF & GABUNGKAN SEMUANYA
# =======================================================
# PERBAIKAN 2: Sesuaikan nama file dengan file PDF aslimu
pdf_faq_path = "../data/custom_data/faq_lengkap.pdf"

print(f"\nMembaca file PDF FAQ dari: {pdf_faq_path}...")

try:
    # 1. Load PDF
    loader = PyPDFLoader(pdf_faq_path)
    pages = loader.load()
    
    # 2. Gabungkan seluruh teks dari semua halaman menjadi satu string panjang
    full_text = "\n".join([page.page_content for page in pages])
    
    # 3. REGEX PINTAR (Regular Expression)
    pattern = r"Q:\s*(.*?)\nA:\s*(.*?)(?=\nQ:|\n\d+\.\s*Intent:|$)"
    matches = re.findall(pattern, full_text, re.DOTALL)
    
    qna_docs = []
    for q, a in matches:
        q_clean = q.strip().replace('\n', ' ')
        a_clean = a.strip().replace('\n', ' ')
        
        content = f"Pertanyaan: {q_clean}\nJawaban: {a_clean}"
        
        doc = Document(
            page_content=content, 
            metadata={"source": "data ML.pdf", "type": "qna_intent"}
        )
        qna_docs.append(doc)

    print(f"[SUKSES] Berhasil mengekstrak {len(qna_docs)} pasang Q&A utuh dan bersih!")
    
    # 4. Intip 2 hasil pertama untuk memastikan formatnya sudah rapi
    if len(qna_docs) > 0:
        print("\n=== CUPLIKAN HASIL EKSTRAKSI Q&A ===")
        print("Chunk 1:")
        print(qna_docs[0].page_content)
        print("\nChunk 2:")
        print(qna_docs[1].page_content)
        print("====================================")
        
    # 5. PERBAIKAN 3: Gabungkan web_docs, qna_docs, dan pdf_docs (Jika Cell 1 sudah dijalankan)
    try:
        semua_dokumen = web_docs + qna_docs + pdf_docs
        print(f"\n[INFO] Total keseluruhan dokumen (54 PDF + Web + FAQ) = {len(semua_dokumen)} chunk.")
    except NameError:
        semua_dokumen = web_docs + qna_docs
        print(f"\n[INFO] Total dokumen sementara (Web + FAQ) = {len(semua_dokumen)} chunk.")
        print("(Catatan: Variabel pdf_docs belum ada karena Cell 1 belum dijalankan, tapi aman untuk dilanjut).")

except Exception as e:
    print(f"Terjadi kesalahan pada ekstraksi FAQ: {e}")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Ditemukan 47 URL. Mulai scraping isi website...
Selesai scraping website!

Membaca file PDF FAQ dari: ../data/custom_data/faq_lengkap.pdf...
[SUKSES] Berhasil mengekstrak 284 pasang Q&A utuh dan bersih!

=== CUPLIKAN HASIL EKSTRAKSI Q&A ===
Chunk 1:
Pertanyaan: Bagaimana  sejarah  Universitas  Halu  Oleo?
Jawaban: Universitas  Halu  Oleo  berasal  dari  Universitas  Halu  Oleo  Swasta  (UNHOL)  yang   didirikan   oleh   Pemerintah   Daerah   Sulawesi   Tenggara   pada   tahun   1964   melalui   Yayasan   Pembina   dan   Pembimbing   Perguruan   Tinggi   Sulawesi   Tenggara   (YP3T).   Pada   tahun   1966   UNHOL   menjadi   cabang   Universitas   Hasanuddin,   kemudian   resmi   menjadi   perguruan   tinggi   negeri   berdasarkan   Keputusan   Presiden   Republik   Indonesia   Nomor   37   Tahun   1981.

Chunk 2:
Pertanyaan: Kapan  UHO  didirikan?
Jawaban: Universitas  Halu  Oleo  resmi  menjadi  perguruan  tinggi  negeri  pada  tanggal  14  Agustus   1981   berdasarkan   Keputusan   P

# Embedding

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import os

# 1. Pisahkan dokumen panjang (Web & PDF) yang butuh di-chunk
# Kita cek apakah pdf_docs sudah ada (jika kamu sudah run Cell 1)
dokumen_panjang = web_docs
if 'pdf_docs' in locals():
    dokumen_panjang += pdf_docs
    print(f"Terdapat {len(pdf_docs)} halaman PDF dan {len(web_docs)} halaman Web yang akan di-chunk.")
else:
    print(f"Hanya terdapat {len(web_docs)} halaman Web yang akan di-chunk.")

# 2. Setup Pemotong Teks (Chunking)
# chunk_size: Maksimal karakter per potongan, chunk_overlap: toleransi irisan agar kalimat tidak hilang konteks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200,
    length_function=len
)

# 3. Lakukan pemotongan teks
print("Mulai memotong dokumen naratif...")
chunked_docs = text_splitter.split_documents(dokumen_panjang)

# 4. Gabungkan chunk naratif dengan dataset Q&A yang utuh
final_docs = chunked_docs + qna_docs
print(f"Total seluruh chunk (Vektor) yang siap masuk database: {len(final_docs)} chunks.")

# 5. Inisialisasi Model Embedding (Kita pakai model open-source gratis yang cepat)
print("\nMenyiapkan model Embedding (mungkin butuh waktu beberapa detik untuk download pertama kali)...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 6. Buat dan Simpan Database Vektor
db_path = "../data/vector_db"
print(f"Memasukkan data ke ChromaDB di folder: {db_path}...")

# Ini akan mengubah semua teks menjadi vektor dan menyimpannya ke folder secara lokal
vectorstore = Chroma.from_documents(
    documents=final_docs,
    embedding=embeddings,
    persist_directory=db_path
)

print("[SUKSES] Vector Database selesai dibuat dan siap digunakan!")

# ==========================================
# 7. TES PENCARIAN (RETRIEVAL) 
# ==========================================
query_test = "Siapa rektor kesatu UHO?"
print(f"\n[TES RAG] Mencari jawaban untuk: '{query_test}'")

# Mencari 2 dokumen paling relevan (k=2)
hasil_pencarian = vectorstore.similarity_search(query_test, k=2)

for i, hasil in enumerate(hasil_pencarian):
    print(f"\n--- Konteks Ditemukan ke-{i+1} ---")
    print(hasil.page_content)

Terdapat 408 halaman PDF dan 455 halaman Web yang akan di-chunk.
Mulai memotong dokumen naratif...
Total seluruh chunk (Vektor) yang siap masuk database: 1473 chunks.

Menyiapkan model Embedding (mungkin butuh waktu beberapa detik untuk download pertama kali)...


C:\Users\HP\AppData\Local\Temp\ipykernel_4260\1661305971.py:33: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Memasukkan data ke ChromaDB di folder: ../data/vector_db...
[SUKSES] Vector Database selesai dibuat dan siap digunakan!

[TES RAG] Mencari jawaban untuk: 'Siapa rektor kesatu UHO?'

--- Konteks Ditemukan ke-1 ---
Pertanyaan: Kontak  resmi  UHO  apa?
Jawaban: Kontak  resmi  UHO  adalah  (0401)  3190105.

--- Konteks Ditemukan ke-2 ---
Pertanyaan: Kontak  resmi  UHO  apa?
Jawaban: Kontak  resmi  UHO  adalah  (0401)  3190105.


# LLM

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Masukkan API Key kamu di sini
os.environ["GOOGLE_API_KEY"] = "API KEY KAMU DI SINI"

print("Menghubungkan ke Google Gemini API...")

# 2. Inisialisasi Model Gemini
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)

# 3. Buat Prompt Template (Instruksi untuk AI)
system_prompt = (
    "Kamu adalah Chatbot cerdas dan ramah untuk layanan informasi FMIPA Universitas Halu Oleo (UHO). "
    "Gunakan potongan konteks berikut untuk menjawab pertanyaan pengguna secara akurat. "
    "Jika jawabannya tidak ada di dalam konteks, katakan saja dengan sopan bahwa kamu tidak memiliki informasi tersebut. "
    "Jangan pernah mengarang informasi (berhalusinasi).\n\n"
    "Konteks:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

# 4. Inisialisasi Retriever dari ChromaDB (Cell 3)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 5. Fungsi kecil untuk mengekstrak dan menggabungkan teks dari dokumen pencarian
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 6. RANGKAI RAG PIPELINE (Menggunakan LCEL - LangChain Expression Language)
# Membaca dari kanan ke kiri/atas ke bawah: 
# Ambil dokumen -> gabungkan -> masukkan ke prompt -> kirim ke LLM -> jadikan teks murni
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("[SUKSES] Otak Chatbot bergaya LCEL berhasil dirakit!")

# ==========================================
# 7. UJI COBA CHATBOT
# ==========================================
print("\n--- MULAI SESI TANYA JAWAB ---")
pertanyaan_user = "Kapan Universitas Halu Oleo didirikan??"

print(f"User: {pertanyaan_user}")
print("Chatbot sedang mencari dan berpikir...")

# Menjalankan rantai RAG
# Karena pakai StrOutputParser, hasilnya langsung berupa teks string!
hasil = rag_chain.invoke(pertanyaan_user)

print(f"\nChatbot FMIPA: {hasil}")

Menghubungkan ke Google Gemini API...
[SUKSES] Otak Chatbot bergaya LCEL berhasil dirakit!

--- MULAI SESI TANYA JAWAB ---
User: Kapan Universitas Halu Oleo didirikan??
Chatbot sedang mencari dan berpikir...

Chatbot FMIPA: Universitas Halu Oleo didirikan pada tahun 1964.


In [5]:
!pip freeze > requirements.txt